<a href="https://colab.research.google.com/github/lilyb838/Final-project/blob/main/Projectv4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rapidfuzz

In [2]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz
from collections import defaultdict
from itertools import combinations
import networkx as nx
from collections import Counter
from torch.utils.data import Dataset
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv('/content/drive/MyDrive/raceform.csv')

/tmp/ipykernel_10342/3453427977.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/raceform.csv')


In [5]:
print(df.shape)

(1851285, 37)


In [6]:
(df.isna().mean() * 100).round(4)

,0
date,0.0000
course,0.0000
race_id,0.0000
off,0.0000
race_name,0.0000
type,0.0000
class,41.5141
pattern,85.7743
rating_band,58.4214
age_band,0.0086


In [8]:
def convert_sp(x):
    if pd.isna(x) or x == '':
        return None

    x = x.replace('F', '').replace('J', '').replace('C', '')

    if x == 'Evens' or x == "Evs":
        return 1.0

    if '/' in x:
        num, den = x.split('/')
        return float(num) / float(den)

    try:
        return float(x)
    except:
        return None

df['sp'] = df['sp'].apply(convert_sp)

In [9]:
df['ovr_btn'] = pd.to_numeric(
    df['ovr_btn'].replace(['-'], np.nan),
    errors='coerce'
)

In [10]:
df = df.rename(columns={"class": "race_class"})
df = df.rename(columns={"type": "race_type"})

In [11]:
#convert distance to furlongs
def convert_dist(x):
  if pd.isna(x):
        return None

  x = str(x).strip()

  miles = re.search(r'(\d+)m', x)
  furlongs = re.search(r'(\d+(?:/\d+)?)f', x)

  total = 0

  if miles:
      total += int(miles.group(1)) * 8

  if furlongs:
      f = furlongs.group(1)
      if '/' in f:
          num, den = f.split('/')
          total += float(num) / float(den)
      else:
          total += float(f)

  return total

df['dist'] = df['dist'].apply(convert_dist)

In [12]:
#extract year and month from date
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

In [13]:
cols_to_drop = list(df.columns[df.isna().mean() > 0.5]) +["comment", "num", "rpr", "ts", "prize", "time", "ran", "race_name", "btn" ]
df = df.drop(columns=cols_to_drop)

In [14]:
df['sp'] = df['sp'].fillna(df['sp'].median())
df['draw'] = df['draw'].fillna('MISSING')
df['draw'] = df['draw'].astype(str)
df['race_class'] = df['race_class'].fillna('MISSING')
df['draw'] = df['draw'].fillna('MISSING')
df['age_band'] = df['age_band'].fillna('MISSING')
df['going'] = df['going'].fillna('MISSING')
df['jockey'] = df['jockey'].fillna('MISSING')
df['trainer'] = df['trainer'].fillna('MISSING')
df['draw'] = df['draw'].fillna('MISSING')
df['dam'] = df['dam'].fillna('MISSING')
df['damsire'] = df['damsire'].fillna('MISSING')
df['owner'] = df['owner'].fillna('MISSING')

In [15]:
df['pos_numeric'] = pd.to_numeric(df['pos'], errors='coerce').fillna(0)
df['finished'] = df['pos_numeric'].notna().astype(int)

In [16]:
# Convert weight from "11-6" format to total pounds
parts = df["wgt"].str.split("-", expand=True)

df["wgt_lbs"] = (parts[0].astype(float) * 14 + parts[1].astype(float))

In [17]:
df['dam_clean'] = df['dam'].str.replace(r'[()]', '', regex=True).str.strip()

In [18]:
df['horse_clean'] = df['horse'].str.replace(r'[()]', '', regex=True).str.strip()

In [19]:
# Estimate birth year from race age
df["birth_year"] = df["date"].dt.year - df["age"]

# Convert to current age
current_year = pd.Timestamp.today().year
df["current_age"] = current_year - df["birth_year"]

In [20]:
df['dam_clean'] = (
    df['dam_clean']
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r'\b(i)\b', '', regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [21]:
df['horse_clean'] = (
    df['horse_clean']
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [22]:
df['sire_clean'] = (
    df['sire']
    .str.replace(r'[()]', '', regex=True).str.strip()
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r'\b(i)\b', '', regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [23]:
df['damsire_clean'] = (
    df['damsire']
    .str.replace(r'[()]', '', regex=True).str.strip()
    .str.lower()
    .str.replace(r'\.', '', regex=True).str.strip()
    .str.replace(r'\b(i)\b', '', regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [25]:
horse_mapping = {
    "angea cara fr": "angela cara fr",
    "hardi blue trois aa fr": "hardi blue trois fr",
    "sublime chope fr": "sublime chop fr"
}
df["horse_final"] = df["horse_clean"].replace(horse_mapping)

In [26]:
damsire_mapping = {
    "a p indy": "ap indy",
    "ut*windsor heights" : "windsor heights",
    "ut*mangarose" : "mangarose",
    "almutawakel" : "almutawakeli"
}
mask = (
    df["dam_clean"].eq("private jet usa")
    & df["damsire_clean"].isna()
)

df.loc[mask, "damsire_clean"] = "smart strike"
df["damsire_clean"] = df["damsire_clean"].str.rstrip()

df["damsire_final"] = df["damsire_clean"].replace(horse_mapping)

In [27]:
dam_mapping = {
    "o k angie arg" : "ok angie arg",
    "moonlight shadow gb" : "moon light shadow gb",
    "sound out ire" : "soundout ire",
    "ticker tape gb" : "ticker tapei gb",
    "ascolini aus" : "ascolini nz",
    "nation ii usa" : "nation usa",
    "sun song ii fr" : "sun song fr"
}
mask = (
    df["horse_clean"].eq("hapi jpn")
    & df["damsire_clean"].isna()
)

df.loc[mask, "dam_clean"] = "queen pirates jpn"

df["dam_final"] = df["dam_clean"].replace(horse_mapping)

In [28]:
df["dam_id"] = (
    df["dam_final"].str.strip().str.replace(r"\s+", "_", regex=True)
    + "_" +
    df["damsire_clean"].fillna("unknown_dam").str.strip().str.replace(r"\s+", "_", regex=True)
)

In [29]:
df["horse_id"] = (
    df["horse_final"].str.strip().str.replace(r"\s+", "_", regex=True)
    + "_" +
    df["dam_id"].fillna("unknown_dam").str.strip().str.replace(r"\s+", "_", regex=True)
)

In [30]:
df["damsire_id"] = (
    df["damsire_final"].str.strip().str.replace(r"\s+", "_", regex=True)
)

In [31]:
df["sire_id"] = (
    df["sire_clean"].str.strip().str.replace(r"\s+", "_", regex=True)
)

In [32]:
def clean_names(name):
    if pd.isna(name):
        return None

    name = name.lower().strip()
    name = re.sub(r"\b(mr|mrs|ms|miss|dr)\b", "", name)
    name = name.replace("'", "")
    name = re.sub(r"[^a-z\s]", " ", name)
    name = " ".join(name.split())

    return name



In [33]:
df["jockey_clean"] = df["jockey"].apply(clean_names)

In [34]:
df["trainer_clean"] = df["trainer"].apply(clean_names)

In [35]:
df["owner_clean"] = df["owner"].apply(clean_names)

In [36]:
jockey_mapping = {
    "E Dwan": "Evan Dwan",
    "M J M OSullivan": "Michael OSullivan"
}
df["jockey_clean"] = df["jockey_clean"].replace(jockey_mapping)

In [37]:
# Names that must keep their titles as separate identities
keep_titles = {
    "mr a jones",
    "miss a jones",
    "miss m osullivan",
    "ms m osullivan"
}


# Restore original titled names for these exceptions
df.loc[
    df["jockey"].str.lower().isin(keep_titles),
    "jockey_clean"
] = df.loc[
    df["jockey"].str.lower().isin(keep_titles),
    "jockey"
].str.lower()

In [38]:
df["jockey_id"] = (
    df["jockey_clean"].str.strip().str.replace(r"\s+", "_", regex=True)
)

In [39]:
# -----------------------------------
# Remove titles from names
# -----------------------------------

def strip_titles(name):

    titles = {"mr", "mrs", "ms", "miss"}

    return [
        x.lower()
        for x in name.split()
        if x.lower() not in titles
    ]



# -----------------------------------
# Match two cleaned names
# -----------------------------------

def matches_name(base_parts, candidate_parts):

    # surname
    if base_parts[-1] != candidate_parts[-1]:
        return False


    # first name
    base_first = base_parts[0]
    candidate_first = candidate_parts[0]


    if len(base_first) == 1:

        if candidate_first[0] != base_first:
            return False

    else:

        if candidate_first != base_first:
            return False



    # middle names

    base_middle = base_parts[1:-1]
    candidate_middle = candidate_parts[1:-1]


    for i, middle in enumerate(base_middle):

        if i >= len(candidate_middle):
            return False


        if len(middle) == 1:

            if candidate_middle[i][0] != middle:
                return False

        else:

            if candidate_middle[i] != middle:
                return False


    return True



# -----------------------------------
# Check whether candidates are
# genuinely conflicting identities
# -----------------------------------

def candidate_group_conflict(candidates):

    cleaned = [
        strip_titles(name)
        for name in candidates
    ]


    for i in range(len(cleaned)):

        for j in range(i + 1, len(cleaned)):

            a = cleaned[i]
            b = cleaned[j]


            # surname
            if a[-1] != b[-1]:
                return True


            # first name
            if (
                len(a[0]) > 1
                and len(b[0]) > 1
                and a[0] != b[0]
            ):
                return True


            # middle names

            middle_a = a[1:-1]
            middle_b = b[1:-1]


            if middle_a and middle_b:

                if len(middle_a) != len(middle_b):
                    return True


                for x, y in zip(middle_a, middle_b):

                    if len(x) > 1 and len(y) > 1:

                        if x != y:
                            return True


                    elif len(x) == 1 and len(y) > 1:

                        if x != y[0]:
                            return True


                    elif len(y) == 1 and len(x) > 1:

                        if y != x[0]:
                            return True


    return False

def match_table(entity):

  entity_clean = f"{entity}_clean"


  # -----------------------------------
  # Store original names for display
  # -----------------------------------

  entity_display = (
      df[[entity_clean, entity]]
      .dropna()
      .drop_duplicates()
      .groupby(entity_clean)[entity]
      .apply(list)
      .to_dict()
  )



  # -----------------------------------
  # Unique cleaned names
  # -----------------------------------

  entity_names = pd.DataFrame({
      entity_clean: df[entity_clean]
      .dropna()
      .unique()
  })


  entity_names["parts"] = (
      entity_names[entity_clean]
      .str.split()
  )


  entity_names = entity_names[
      entity_names["parts"].str.len() >= 2
  ].copy()


  entity_names["surname"] = (
      entity_names["parts"]
      .str[-1]
  )



  # -----------------------------------
  # Group by surname
  # -----------------------------------

  surname_groups = defaultdict(list)


  for row in entity_names.itertuples(index=False):

      surname_groups[row.surname].append(
          (
              getattr(row, entity_clean),
              row.parts
          )
      )

      # -----------------------------------
  # Find possible matches
  # -----------------------------------

  results = []


  for row in entity_names.itertuples(index=False):

      base_name = getattr(row, entity_clean)
      base_parts = row.parts
      surname = row.surname


      candidates = []


      for candidate_name, candidate_parts in surname_groups[surname]:

          if candidate_name == base_name:
              continue

          if matches_name(base_parts, candidate_parts):
              candidates.append(candidate_name)



      candidates = sorted(set(candidates))



      # -----------------------------------
      # Keep only genuine conflicts
      # -----------------------------------

      if (
          len(candidates) >= 2
          and candidate_group_conflict(candidates)
      ):


          base_display = entity_display.get(
              base_name,
              [base_name]
          )


          candidate_display = []

          for candidate in candidates:

              candidate_display.extend(
                  entity_display.get(
                      candidate,
                      [candidate]
                  )
              )

          candidate_display = sorted(
              set(candidate_display)
          )

          results.append({
              "base_name": ", ".join(base_display),
              "n_candidates": len(candidate_display),
              "candidates": ", ".join(candidate_display)

          })



  # -----------------------------------
  # Initial ambiguity table
  # -----------------------------------

  final_match_table = pd.DataFrame(results)
  return final_match_table

In [40]:
def overlap(entity, final_match_table):

  entity_clean = f"{entity}_clean"

  entity_original_to_clean = (
      df[[entity, entity_clean]]
      .dropna()
      .drop_duplicates()
      .set_index(entity)[entity_clean]
      .to_dict()
  )


  # -----------------------------------
  # entity_clean -> horse_clean set
  # -----------------------------------

  entity_horses = (
      df.dropna(subset=[entity_clean, "horse_id"])
        .groupby(entity_clean)["horse_id"]
        .apply(set)
        .to_dict()
  )


  # -----------------------------------
  # entity name -> horses
  # -----------------------------------

  def entity_to_horses(name):

      clean_name = entity_original_to_clean.get(name)

      if clean_name is None:
          return set()

      return entity_horses.get(clean_name, set())



  # -----------------------------------
  # Find overlaps
  # -----------------------------------

  def find_overlap_pairs(row):

      base = row["base_name"]
      candidates = row["candidates"].split(", ")

      base_candidate_pairs = []
      candidate_candidate_pairs = []


      # -------------------------------
      # Base vs candidate
      # -------------------------------

      for candidate in candidates:

          overlap = (
              entity_to_horses(base)
              .intersection(
                  entity_to_horses(candidate)
              )
          )

          if overlap:
              base_candidate_pairs.append(
                  f"{base} - {candidate}"
              )


      # -------------------------------
      # Candidate vs candidate
      # -------------------------------

      for a, b in combinations(candidates, 2):

          overlap = (
              entity_to_horses(a)
              .intersection(
                  entity_to_horses(b)
              )
          )

          if overlap:
              candidate_candidate_pairs.append(
                  f"{a} - {b}"
              )


      return pd.Series({
          "base_candidate_overlap_pairs": "; ".join(base_candidate_pairs),
          "candidate_candidate_overlap_pairs": "; ".join(candidate_candidate_pairs)
      })



  # -----------------------------------
  # Add columns
  # -----------------------------------

  entity_overlap_table = (
      final_match_table
      .copy()
  )


  entity_overlap_table[
      [
          "base_candidate_overlap_pairs",
          "candidate_candidate_overlap_pairs"
      ]
  ] = (
      entity_overlap_table
      .apply(find_overlap_pairs, axis=1)
  )

  return entity_overlap_table

In [41]:
def map_overlap(entity, candidate_table):

  entity_overlap_table = candidate_table.copy()

  G = nx.Graph()

  for _, row in entity_overlap_table.iterrows():

      pairs = []

      if row["base_candidate_overlap_pairs"]:
          pairs.extend(
              row["base_candidate_overlap_pairs"].split("; ")
          )

      if row["candidate_candidate_overlap_pairs"]:
          pairs.extend(
              row["candidate_candidate_overlap_pairs"].split("; ")
          )


      for pair in pairs:

          entity1, entity2 = pair.split(" - ")

          G.add_edge(entity1, entity2)



  # -----------------------------------
  # Count original entity appearances
  # -----------------------------------

  entity_counts = (
      df[entity]
      .value_counts()
      .to_dict()
  )


  # -----------------------------------
  # Map each connected group to
  # most common entity name
  # -----------------------------------

  entity_mapping = {}

  for component in nx.connected_components(G):

      canonical_name = max(
          component,
          key=lambda x: entity_counts.get(x, 0)
      )

      for entity in component:
          entity_mapping[entity] = canonical_name

  return entity_mapping

In [42]:
trainer_table = match_table("trainer")
trainer_overlap = overlap("trainer", trainer_table)
trainer_map = map_overlap("trainer", trainer_overlap)
df["trainer_id"] = (
    df["trainer"]
    .replace(trainer_map)
    .str.lower()
    .str.replace(" ", "_")
)

In [43]:
owner_table = match_table("owner")
owner_overlap = overlap("owner", owner_table)
owner_map = map_overlap("owner", owner_overlap)
df["owner_id"] = (
    df["owner"]
    .replace(trainer_map)
    .str.lower()
    .str.replace(" ", "_")
)

In [44]:
df = df.sort_values(['horse_id', 'date'])

df['horse_ewa_pos'] = (
    df.groupby('horse_id')['pos_numeric']
      .shift()
      .groupby(df['horse_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)

df['horse_ewa_btn'] = (
    df.groupby('horse_id')['ovr_btn']
      .shift()
      .groupby(df['horse_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)
df['horse_ewa_finsih_rate'] = (
    df.groupby('horse_id')['finished']
      .shift()
      .groupby(df['horse_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)

In [45]:
df = df.sort_values(['jockey_id', 'date'])

df['jockey_ewa_pos'] = (
    df.groupby('jockey_id')['pos_numeric']
      .shift()
      .groupby(df['jockey_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)
df['jockey_ewa_btn'] = (
    df.groupby('jockey_id')['ovr_btn']
      .shift()
      .groupby(df['jockey_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)
df['horse_ewa_finsih_rate'] = (
    df.groupby('jockey_id')['finished']
      .shift()
      .groupby(df['jockey_id'])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=0, drop=True)
)

In [46]:
df['horse_jockey_ewa_pos'] = (
    df.groupby(['horse_id', 'jockey_id'])['pos_numeric']
      .shift()
      .groupby([df['horse_id'], df['jockey_id']])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=[0,1], drop=True)
)
df['horse_jockey_ewa_btn'] = (
    df.groupby(['horse_id', 'jockey_id'])['ovr_btn']
      .shift()
      .groupby([df['horse_id'], df['jockey_id']])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=[0,1], drop=True)
)
df['horse_jockey_ewa_finish_rate'] = (
    df.groupby(['horse_id', 'jockey_id'])['finished']
      .shift()
      .groupby([df['horse_id'], df['jockey_id']])
      .ewm(alpha=0.3, adjust=False)
      .mean()
      .reset_index(level=[0,1], drop=True)
)

In [48]:
df["top3"] = (df["pos_numeric"] <= 3).astype(float)

In [49]:
ignore_cols = ['horse', 'dam', 'sire', 'damsire', 'jockey', 'trainer', 'owner', 'pos', 'date', 'wgt', 'jockey_clean', 'trainer_clean', 'owner_clean', 'horse_clean', 'dam_clean', 'damsire_clean', 'sire_clean','dam_final', 'horse_final', 'damsire_final' , 'draw', 'off', 'or', 'pos_numeric', 'ovr_btn']
target_col = ['top3']
cat_cols = ['horse_id', 'dam_id', 'sire_id', 'damsire_id', 'jockey_id', 'trainer_id', 'owner_id', 'course', 'race_class', 'month', 'race_type', 'sex', 'going', 'age_band',]
num_cols = [c for c in df.columns if c not in cat_cols + ignore_cols + target_col]
#clean up the horse_clean / horse_final stuff and draw, off, or, pos_numeric, ovr_btn

In [58]:
y = df[target_col].copy()
d = pd.to_numeric(df["ovr_btn"], errors="coerce").fillna(0)
w = torch.where(
    torch.tensor(y.values.squeeze(), dtype=torch.float32) == 1,
    torch.tensor(1.0),
    1 - torch.exp(-0.3 * torch.tensor(d.values, dtype=torch.float32))
)
def weighted_BCE_loss(p, y, w):
    loss = - (w * y * torch.log(p) + w * (1 - y) * torch.log(1 - p))
    return loss.mean()

In [59]:
# Sort chronologically first
df = df.sort_values("date").reset_index(drop=True)

# Create feature sets
X_cat = df[cat_cols].copy()
X_num = df[num_cols].copy()

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.90)

w_train = w[:train_end]
w_val = w[train_end:val_end]
w_test = w[val_end:]

# Train
X_cat_train = X_cat.iloc[:train_end].copy()
X_num_train = X_num.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()


# Validation
X_cat_val = X_cat.iloc[train_end:val_end].copy()
X_num_val = X_num.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()


# Test
X_cat_test = X_cat.iloc[val_end:].copy()
X_num_test = X_num.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

X_num_train = X_num_train.fillna(0)
X_num_val = X_num_val.fillna(0)
X_num_test = X_num_test.fillna(0)

scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

cat_maps = {}

for col in cat_cols:

    # Fit mapping only on training data
    labels, uniques = pd.factorize(
        X_cat_train[col]
    )
    cat_maps[col] = uniques

    mapping = {
        value: idx
        for idx, value in enumerate(uniques)
    }

    X_cat_train[col] = labels
    X_cat_val[col] = X_cat_val[col].map(mapping)
    X_cat_test[col] = X_cat_test[col].map(mapping)
    X_cat_val[col] = X_cat_val[col].fillna(0)
    X_cat_test[col] = X_cat_test[col].fillna(0)

X_cat_train = torch.tensor(
    X_cat_train.values,
    dtype=torch.long
)

X_cat_val = torch.tensor(
    X_cat_val.values,
    dtype=torch.long
)

X_cat_test = torch.tensor(
    X_cat_test.values,
    dtype=torch.long
)


X_num_train = torch.tensor(
    X_num_train,
    dtype=torch.float32
)

X_num_val = torch.tensor(
    X_num_val,
    dtype=torch.float32
)

X_num_test = torch.tensor(
    X_num_test,
    dtype=torch.float32
)


y_train = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

y_val = torch.tensor(
    y_val.values,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test.values,
    dtype=torch.float32
)

In [60]:
class RacingDataset(Dataset):
    def __init__(self, X_cat, X_num, y, w):
        self.X_cat = X_cat
        self.X_num = X_num
        self.y = y
        self.w = w

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_cat[idx],
            self.X_num[idx],
            self.y[idx],
            self.w[idx]
        )

In [61]:
#based off coursework code
train_dataset = RacingDataset(
    X_cat_train,
    X_num_train,
    y_train,
    w_train
)

val_dataset = RacingDataset(
    X_cat_val,
    X_num_val,
    y_val,
    w_val
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

In [62]:
#based off rugby code
class HorsePredictionMLP(nn.Module):
    def __init__(self, n_horse_id, n_dam_id, n_sire_id, n_damsire_id, n_jockey_id, n_trainer_id, n_owner_id, n_course, n_race_class, n_month, n_race_type, n_sex, n_going, n_age_band, num_features, output_size, dropout=0.2, hidden_size=512):
        super().__init__()
        self.horse_id_emb = nn.Embedding(n_horse_id, 32)
        self.jockey_id_emb = nn.Embedding(n_jockey_id, 8)
        self.trainer_id_emb = nn.Embedding(n_trainer_id, 8)
        self.owner_id_emb = nn.Embedding(n_owner_id, 8)

        self.sire_id_emb = nn.Embedding(n_sire_id, 8)
        self.dam_id_emb = nn.Embedding(n_dam_id, 16)
        self.damsire_id_emb = nn.Embedding(n_damsire_id, 8)

        self.course_emb = nn.Embedding(n_course, 8)
        self.race_type_emb = nn.Embedding(n_race_type, 3)
        self.race_class_emb = nn.Embedding(n_race_class, 3)
        self.age_band_emb = nn.Embedding(n_age_band, 3)
        self.going_emb = nn.Embedding(n_going, 4)
        self.month_emb = nn.Embedding(n_month, 4)
        self.sex_emb = nn.Embedding(n_sex, 2)

        total_emb = (32 + 8 + 8 + 8 + 8 + 16 + 8 + 8 + 3 + 3 + 3 + 4 + 4 + 2)

        self.fc = nn.Sequential(
          nn.Linear(total_emb + num_features, 512), #do i keep hdden_size = 512?
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(512, 256),
          nn.ReLU(),
          nn.Dropout(dropout),
          nn.Linear(256, 128),
          nn.ReLU(),
          nn.Linear(128, output_size),
          nn.Sigmoid()
        )

    def forward(self, batch_cat, X_num):

      horse_id = batch_cat[:, 0]
      dam_id = batch_cat[:, 1]
      sire_id = batch_cat[:, 2]
      damsire_id = batch_cat[:, 3]
      jockey_id = batch_cat[:, 4]
      trainer_id = batch_cat[:, 5]
      owner_id = batch_cat[:, 6]
      course = batch_cat[:, 7]
      race_class = batch_cat[:, 8]
      month = batch_cat[:, 9]
      race_type = batch_cat[:, 10]
      sex = batch_cat[:, 11]
      going = batch_cat[:, 12]
      age_band = batch_cat[:, 13]

      emb = torch.cat([
          self.horse_id_emb(horse_id),
          self.dam_id_emb(dam_id),
          self.sire_id_emb(sire_id),
          self.damsire_id_emb(damsire_id),
          self.jockey_id_emb(jockey_id),
          self.trainer_id_emb(trainer_id),
          self.owner_id_emb(owner_id),
          self.course_emb(course),
          self.race_class_emb(race_class),
          self.month_emb(month),
          self.race_type_emb(race_type),
          self.sex_emb(sex),
          self.going_emb(going),
          self.age_band_emb(age_band)
      ], dim=1)

      x = torch.cat([emb, X_num], dim=1)

      return self.fc(x)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

In [63]:
model_args = {
    "n_horse_id": len(cat_maps["horse_id"]),
    "n_dam_id": len(cat_maps["dam_id"]),
    "n_sire_id": len(cat_maps["sire_id"]),
    "n_damsire_id": len(cat_maps["damsire_id"]),
    "n_jockey_id": len(cat_maps["jockey_id"]),
    "n_trainer_id": len(cat_maps["trainer_id"]),
    "n_owner_id": len(cat_maps["owner_id"]),
    "n_course": len(cat_maps["course"]),
    "n_race_class": len(cat_maps["race_class"]),
    "n_month": len(cat_maps["month"]),
    "n_race_type": len(cat_maps["race_type"]),
    "n_sex": len(cat_maps["sex"]),
    "n_going": len(cat_maps["going"]),
    "n_age_band": len(cat_maps["age_band"]),
    "num_features": X_num_train.shape[1],
    "output_size": y_train.shape[1]
}
mlp_model = HorsePredictionModel(
    **model_args,
    dropout=0.2,
).to(device)


num_epochs = 20
batch_size = 1024

mlp_model.to(device)

HorsePredictionModel(
  (horse_id_emb): Embedding(157121, 32)
  (jockey_id_emb): Embedding(6571, 8)
  (trainer_id_emb): Embedding(9285, 8)
  (owner_id_emb): Embedding(74208, 8)
  (sire_id_emb): Embedding(4757, 8)
  (dam_id_emb): Embedding(81634, 16)
  (damsire_id_emb): Embedding(5463, 8)
  (course_emb): Embedding(383, 8)
  (race_type_emb): Embedding(4, 3)
  (race_class_emb): Embedding(8, 3)
  (age_band_emb): Embedding(27, 3)
  (going_emb): Embedding(21, 4)
  (month_emb): Embedding(12, 4)
  (sex_emb): Embedding(8, 2)
  (fc): Sequential(
    (0): Linear(in_features=132, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

In [64]:
#based off coursework code
def train_and_validate(model, train_loader, val_loader, optimizer, device):

  train_losses = []
  val_losses = []
  all_val_probs = []
  all_val_labels = []
  num_epochs = 20
  patience = 5
  epochs_no_improve = 0
  min_val_loss = float('inf')

  for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_cat, batch_num, batch_y , batch_w in train_loader:

        batch_cat = batch_cat.to(device)
        batch_num = batch_num.to(device)
        batch_y = batch_y.to(device)
        batch_w = batch_w.to(device)

        optimizer.zero_grad()
        p = model(batch_cat, batch_num)

        loss = weighted_BCE_loss(
            p,
            batch_y,
            batch_w
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)

  #validation

    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
      for batch_cat, batch_num, batch_y, batch_w in val_loader:

        batch_cat = batch_cat.to(device)
        batch_num = batch_num.to(device)
        batch_y = batch_y.to(device)
        batch_w = batch_w.to(device)

        p = model(batch_cat, batch_num)

        loss = weighted_BCE_loss(
            p,
            batch_y,
            batch_w
        )

        running_loss += loss.item()

        all_val_probs.extend(p.cpu().numpy())
        all_val_labels.extend(batch_y.cpu().numpy())

      epoch_val_loss = running_loss / len(val_loader)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    if epoch_val_loss < min_val_loss:
      min_val_loss = epoch_val_loss
      epochs_no_improve = 0
    else:
      epochs_no_improve += 1

    if epochs_no_improve >= patience:
      print(f"Early stopping at epoch {epoch+1}")
      break

    print(f'Epoch {epoch+1}/{num_epochs} '
          f'Train Loss: {epoch_train_loss:.4f}'
          f'Val Loss: {epoch_val_loss:.4f}')

  return train_losses, val_losses

In [65]:
#plot loss and accuracy
def plot_loss_accuracy(train_losses, val_losses):
  plt.figure(figsize=(10, 5))
  plt.plot(train_losses, label = 'Train Loss', marker = 'o')
  plt.plot(val_losses, label = 'Validation Loss', marker = 'o')
  plt.title('Training vs Validation Loss')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.legend()
  plt.grid(True)
  plt.show()

In [66]:
for name, param in mlp_model.named_parameters():
    if torch.isnan(param).any():
        print("NaN parameter:", name)

In [67]:
#Run and plot model
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)
train_losses, val_losses = train_and_validate(
    mlp_model, train_loader, val_loader, optimizer, device)
plot_loss_accuracy(train_losses, val_losses)

Epoch 1/20 Train Loss: 0.5987Val Loss: 0.5929


KeyboardInterrupt: 